# Глава 2. Эмбединги и первые нейросетевые модели
В области обработки естественного языка компьютеры уже умели немало: искать документы по ключевым словам, считать частоты, предсказывать следующее слово по статистике коротких цепочек. Но принципиальной проблемой оставалось то, что для машины слова «кошка» и «кот» были так же далеки друг от друга, как «кошка» и «трактор»: две строки либо совпадают, либо нет, не существует промежуточного состояния. Потребность создать такое представления, слово превратилось из просто "символьной метки" в точку непрерывного векторного пространства смыслов, в котором геометрическая близость отражает близость смысловую.

В этой главе рассмотрим модель Бенжио, первую нейросетевую модель языка. Обсудтим ставшую революционной модель построения статических эмбедингов Word2Vec и её более поздние модификации GloVe и FastText.

## Распределенные представления
До 2000 года задачи языкового моделирования (предсказать следующее слово по предыдущим) решались в основном с помощью n-граммных моделей, которые мы подробно рассматривали в главе I. На протяжении нескольких десятилетий они был State-of-the-Art инструментом в индустрии и его было вполне достаточно для решения некоторых базовых задач, таких как автозаполнение, автокоррекция, разметка последовательностей. Короче, тех задач, где для хорощего уровня ответа достаточно небольшого локального контекста. Но в задачах, требующих более глубокого понимания текста - а это и машинный перевод, и распознавании речи, и задачи генерации - очевидно этого было недостаточно.

Мы уже рассматривали ключевые ограничения n-граммных моделей. Напомним их еще раз. Главным является то, что число всех возможных комбинаций из $n$ слов астрономическое. И соотвественно таким же ялвяется количество вероятностей, которые нужно оценить - при размере словаря в $|V|$ слов требуется вычисление ${|V|}^n$ оценок. Это значит, что даже при использовании скромного словаря в 10 тысяч слов, уже для трехграммной модели (n=3) нужен рассчет триллиона возможных комбинаций. Причем ввиду естественной ограниченности выборки большинство из этих комбинаций ни разу не встречается в обучающих данных, а значит численно оценить их вероятность просто невозможно и приходится "сооружать" искусственные оценки, прибегая к различным инженерным приемам типа сглаживания.

Во-вторых, в рамках n-граммной языковой модели слово - это просто дискретный набор символов. Он лишен какой-либо семантики. «Кот» и «собака» различаются ровно так же, как «кот» и «пылесос», это просто разные индексы в словаре. Из-за этой "дискретности" модель не имеет способности нормально обобщать входные данные и может только их запоминать. При том, что навык обобщения входных данных (generalization) - критичный для любого более или менее серьезного моделирования. Это не просто вопрос эффективной работы с синонимичными терминами, это вопрос отнесения слова к нужной семантической категории, в итоге понимания текста.

Обе проблемы были бы решены, если бы в представление текста $x$ мы как-то добавили свойство "непрерывности", то есть сделали так, чтобы близкие по смыслу слова кодировались "близкими" векторами. Сама идея параметризованного представления существовала ещё с 1984 года, когда __Джефри Хинтон__ в свои работах ввел понятие *"распределенного представления"* (distributed representation) в противовес *"локальному представлению"* (local representation). Если локальное представление это просто назначение метки объекту, например, one-hot кодирование, то распредленное представиление это смесь сразу всех компонентов. Несмотря на кажущуюся неоптимальность такого кодирования - ведь из этого "компота" признаков модели приходится как-то доставать нужный сигнал - оно достаточно эффективно, по-крайней мере представления во всех современных моделях устроены ровно по этому принципу. Главное, что дает такая непрерывность - модель получает способность обобщать: если в ходе обучения «кот» и «собака» получают близкие векторы, то модель, видевшая «кот сидел на полу», автоматически назначит разумную вероятность фразе «собака сидела на полу». 

Ранее в индустрии были попытки создать такие представления. Были семантические графы, построенные по базе синонимов WordNet, где близость двух слов опредеялась как пути на графе зависимостей. В информационном поиске было латентно-семантическое индексирование (LSI), о котором мы говорили ранее, где распределенные представления слова и документа получали через разложение матрицы частот слов в корпусе текстов. Но во-первых матричное разложение не самый удобный способ генерации представлений. А кроме того, в LSI моделировалась именно "тематическая" близость, никак не учитывающая порядок слов. Для поиска документов по теме - это ok, но для задач языкового моделирования полезнее иметь именно "контекстную" близость.

## Модель Бенжио
В 2003 году [(Benjio et al)](https://jmlr.org/papers/volume3/tmp/bengio03a.pdf) хоте в поиске такого представления решили пойти по наиболее гибкому из возможных путей - они просто сопоставли каждому слову из словаря $t \in V$ некоторый обучаемый вещественный вектор $x_t$ (порядка 30–100 измерений), который выполнял роль его непрерывного представления, а в качестве модели использовали нейронную сеть $P(x_t) = F_{net}(x_{t-1}, x_{t-2} , x_{t-n})$. То есть векторное представние слова они по сути отождествили с параметром модели $x_t = \theta_t$, сделав подбор подходящего представления заботой модели. И это очень хорошо ложилось на концепцию нейронных сетей, где традиционно генерацией признаков занимается не разработчик, а сама модель.

Фактически это был один из первых примеров использования *обучаемых эмбедингов*. Способ, который в дальнейшем стал стандартом в модедировании текстовых данных, и который используется до сих пор во всех больших языковых моделях. Сам термин "эмбединг", однако, вошел в обиход позже, после выхода модели Word2Vec, о которой мы поговорим ниже. Заслуга Бенжио в том, что они демокртизировали механизм генерации таких эмбедингов и поместили его в задачу языкового моделирования.

<img src="img/benjio.png" width=400>

Архитекутура модели Бенжио была несложной, она представляла собой стандартную полносвязную нейросеть, состоящую из двух слоев. На первом слое все эмбединги $C(w_{t-n+1}), \dots, C(w_{t-1})$ конкатенируются в единый вектор $x$, на втором проходят через слой с нелинейной функцией активации (гиперболический тангенс). На выходе выполняется softmax по всему словарю $V$, что вместо ничем не ограниченных скоров даёт полноценное распределение вероятностей, суммирующееся в единицу:

$$P(w_t \mid w_{t-1}, w_{t-2}, \dots, w_{t-n+1}) = \frac{e^{y_{w_t}}}{\sum_{i \in V} e^{y_i}}, \qquad y = b + W x + U \tanh(d + H x)$$

Здесь $y_i$ — логит, ненормированный скор для $i$-го слова словаря. Параметрами модели являются матрицы $H, U, W$, векторы смещений $b, d$ и собственно $X$ словарь эмбедингов. Обучается модель через максимизацию логарифма правдоподобия текстов обучающей выборки. Для большей борьбы с запоминанием в функцию потерь может добавляться с регуляризация:

$$\mathcal{L}_B = - \frac{1}{T}\sum_{t} \log \hat{P}(w_t \mid w_{t-n+1}, \dots, w_{t-1}) \longrightarrow \min$$

При всех плюсах у модели оставалась пара фундаментальных узких мест. Во-первых, модель использует контекст фиксированной и как правило небольшой длины (обычно всего несколько токенов), что естественно сильно ограничивает её возможности. Для обхода этого ограничения нужно либо возвращаться к глобальным bag-of-words моделям (при этом перестает учитываться порядок слов), либо использовать рекурсивные сети (при этом на больших контекстах сигнал передается с потерями). О рекуррентных сетях будем говорить в следующей главе.

Во-вторых, вычисление знаменателя в формуле софтмакса крайне дорогостоящая операция из-за того, что необходимо каждый раз считать сумму по всему словарю, который в совеременных моделях может содержать десятки и даже сотни тысяч слов. Это дополнительное большое вычисление на каждом шаге обучения и инференса:
$$P = \text{softmax}(y_i) = \frac{e^{\text{y}_i}}{e^{\text{y}_1} + e^{\text{y}_2} + ... + e^{\text{y}_{|V|}}}$$

## Расчет тяжелого софтмакса
В первой версии модели авторы боролись с этой проблемой просто распараллеливанием подсчета по нескольким процессорам, но в более поздних модификациях стали искать более умный подход и заменили софтмакс на приближенное вычисление. Ниже рассмотрим несколько инженерных приемов, которые помогут частично обойти эту проблему и ускорить вычисления софтмакса.

Стоит отметить, что приёмы, описанные ниже, как правило используются только на этапе обучения. На этапе инференса предпочтение чаще отдают полному расчету вероятностей с честным проходом по всем токенам. Во-первых, цена ошибки на генерации обычно выше. Во-вторых на инфренсе ранжирование токенов обычно важнее хорошо откалиброванных абсолютных вероятностей, и такой порядок ещё более чувствителен к шуму. 

### Иерархический софтмакс
Это подход, предложенный самими авторами [(Bengio et al, 2005)](https://proceedings.mlr.press/r5/morin05a.html) двумя годами позже выхода модели. Вместо того, чтобы вычислять вероятности для всех слов выходного слоя (что требует $O(|V|)$ операций), метод строит бинарное дерево, где листьями являются слова, а внутренними узлами — бинарные классификаторы. Вероятность конкретного слова вычисляется как произведение вероятностей прохождения по пути от корня к соответствующему листу, где на каждом узле принимается решение (левая или правая ветвь) с использованием сигмоид-функции. Это снижает вычислительную сложность с $O(|V|)$ до $O(log |V|)$ (средняя глубина дерева), что позволяет обучать модели на миллионных словарях, однако на практике качество такой аппроксимации может немного уступать полному softmax, а её производительность сильно зависит от структуры дерева (часто используется дерево Хаффмана для ускорения на частых словах), и в современных архитектурах она всё чаще уступает место другим методам вроде негативного сэмплирования, оставаясь при этом фундаментальным алгоритмом в истории NLP.

Вместо проекции в словарь обучается бинарное дерево, по которому восстанавливается вероятность токена при заданном конексте h. 

Идея иерархического softmax: параметризуем не слова, а внтуренние ноды дерева поиска (листья - слова). Каждое такое представление внутренней ноды перенаправляет $h$ влево или вправо. Таким образом для подсчета вероятности $P(w|h)$ нужно сделать $log(|V|)$ шагов. В модели Bengio дерево строили по WordStat.

### Сэмплирование по важности
Сэмплирование по важности (importance sampling) - это классический прием из математической статистики, известный ещё с 1940-х годов. Его используют для сэмплирования наблюдений из некоторого *сложного* распределения, когда *честное* сэмплирование трудоемко. "Сложное распредление" в данном случае это распределение, требующее вычисления интеграла $P$, на всех точках, где оно задано. В основе метода простая идея, что сэмплирование из сложного распределения $P$ можно приближенно заменить на сэмплирование из более простого распределения $Q$ (например, равномерного), если добавить корректирующую поправку. В нашем случае распределение.

#### Математическое обоснование
Допустим, мы насэмплировали из равномерного распредления множество слов $S$ (например, 100–200 штук). Главное что нужно понять: если мы посчитаем знаменатель $Z$ не на всех, а только на этих примерах, будет ли такая оценка софтмакса несмещенной? Из курса статистики мы знаем, что выборочное среднее $\vec{X} = \frac{1}{N} \sum x$ является несмещенной оценкой. И соотвественно выборочная сумма $N \cdot \vec{X}$ тоже. То есть, что касается знаменателя, здесь ответ утвердительный, его выборочная оценка действительно несмещенная: 
$$\mathrm{E}(Z_S) = |V| \cdot \mathbb{E} \bigg( \frac{1}{|S|} \sum e^{y_i} \bigg) = Z$$

Но вот деление $1/Z_S$, которое нам нужно для вычисления знаменателя $e^y/Z_S$, немного портит картину, поскольку модифицирует $Z_S$ неравномерно. Левый хвост распределения $Z_{S}$ оно быстро поднимает, причем в бесконечность, а правый хвост умеренно гасит. В итоге масса вероятности перераспределяется в большую сторону и вероятность в среднем получается завышенной. Причем, чем шире разброс $Z_{S}$, тем больше смещение.
$$
\mathbb{E}[\hat{p}_i] = \mathbb{E}\left[\frac{e^{y_i}}{Z_{\text{S}}}\right] = e^{y_i} \cdot \mathbb{E}\left[\frac{1}{Z_{\text{S}}}\right] \geqslant e^{y_i} \cdot \frac{1}{\mathbb{E}[Z_{\text{S}}]} = \frac{e^{y_i}}{Z} = p_i
$$

В методе Importance Sampling каждое наблюдение взвешивается с коэффициентом, обратным вероятности его попадания в выборку. Если слово $j$ имеет высокую вероятность $q(j)$ быть выбранным, то оно появляется в выборке часто, и его вклад в знаменатель нужно уменьшить. Если слово редко попадается в выборке (низкое $q(j)$), то его случайное появление в $S$ — событие значимое, и мы должны увеличить его вклад.

$$
Z_{IS} = \frac{1}{|S|} \sum_{j \in S} \frac{e^{y_j}}{q(j)}
$$

Легко проверить, что  оценка $\hat{Z}_{\text{IS}}$ также несмещена. Но теперь — ключевой момент — дисперсия этой оценки. Если выбрать $q(j) \propto e^{y_j}$ (оптимальное распределение), то Var = 0. На практике мы не знаем $e^{y_j}$ заранее (мы же их и пытаемся оценить), поэтому не можем использовать оптимальное $q$. Но мы можем выбрать $q$, близкое к $e^{y_j}$ — например, пропорциональное частоте слов в корпусе. Чем ближе $q$ к оптимальному, тем меньше дисперсия.

### Выборчный софтмакс
Хотя оценка знаменателя в Importance Sampling получается несмещённой, сама вероятность $\frac{e^{y_i}}{Z}$ (отношение) при подстановке приближённого $Z$ становится смещённой. Из-за нелинейности операции деления ошибки в знаменателе неравномерно влияют на итоговую вероятность. Чтобы обойти это, в Sampled Softmax заменяют не вычисление знаменателя, а саму функцию потерь.

Идея заимствована из __NCE__ (Noise Contrastive Estimation): вместо того, чтобы учить модель предсказывать правильную категорию среди тысяч, мы формулируем задачу как бинарную классификацию. Для каждого обучающего примера (пары «контекст → целевое слово») мы берём один положительный пример — само целевое слово $w_t$ и $k$ отрицательных (шумовых) примеров — случайных слов из словаря, взятых из некоторого простого распределения (например, равномерного или униграммного).

Теперь модель обучается не предсказывать распределение по $|V|$ классам, а отличать правильный токен от случайных «шумовых» подделок. Для положительного примера мы хотим, чтобы выход сети (скалярное произведение или логит) был большим, а для отрицательных — маленьким. Функция потерь для одного примера выглядит так:

$$
\mathcal{L} = - \log \sigma(y_{w_t}) - \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_{noise}} \left[ \log \sigma(-y_{w_i}) \right]
$$

где $\sigma$ — сигмоида, а $y_w$ — выход сети для слова $w$.

Почему это работает и чем отличается от IS? Вместо того чтобы приближать знаменатель, мы переформулируем саму цель обучения. Оказывается, что если взять $k$ достаточно большим (обычно 5–25 для больших словарей), то градиенты этой бинарной функции потерь с высокой точностью аппроксимируют градиенты полного softmax. При этом мы **вообще не вычисляем** знаменатель, а работаем только с $k+1$ словами. Это не только ускоряет вычисления, но и часто даёт более стабильное обучение, чем IS, потому что мы не делим на шумную оценку $Z$ и не боремся с проблемой огромной дисперсии, характерной для сэмплирования по важности на редких словах. Именно эта логика (в варианте *negative sampling*) позже сделает модель Word2Vec невероятно быстрой и масштабируемой.

## Модель Word2Vec
В 2013 году [(Mikolov et al.)](https://arxiv.org/abs/1301.3781) с коллегами из Google задался вопросом: если нужны только векторы слов, зачем обучать полноценную языковую модель? Так появилось переосмсыление модели Бенжио, модель **Word2Vec**. Модель была намеренно упрощена: из неё выброшен дорогой скрытый нелинейный слой и осталась только линейная проекция.

Контекстом слова длины $c$ будем называть все слова, находящиеся слева и справа от него на расстоянии не больее $c$. Возьмем для примера фразу $\text{"Я люблю пить горячий чай с лимоном"}$. Для слова "горячий" контекстом длины 2 будет множество слов (люблю, пить, чай, с). Для слова "чай" (пить, горячий, с, лимоном).

Сформулировал две альтерантных постановки. Первую он назвал __CBOW__ (continuous bag-of-words), в рамках которой модель учится предсказывать центральное слово по его контексту, а вторую модель __Skip-gram__, в рамках которой модель учится по центральному слову предсказывать контекст. Поскольку подходы давали сопоставимые по качеству результаты, в своей работе авторы приводят оба. Skip-gram обычно генерирует лучшие справляется с  редкими словами, но CBOW всреднем работает быстрее.

В постановке Skip-gram максимизируется среднюю логарифмическую вероятность всех контекстных слов $w_{t+j}$ при данном центральном слове $w_t$ в окне радиуса $c$:

$$\mathcal{L}_{WV} = -\frac{1}{T}\sum_{t=1}^{T} \sum_{-c \le j \le c,\; j \ne 0} \log p(w_{t+j} \mid w_t) \rightarrow \min$$

В случае с CBOW постановкой задачи максимизируемая вероятность меняется на соотвественно $P(w_{t} | w_{t-c} ... w_{t+c})$.

Word2Vec моделирует контекстную релевантность пары слов как скалярное произведение двух векторов — вектора $\mathbf{v}_w$ для центрального слова и вектора $\mathbf{u}_w$ для контекстного слова. А софтмакс поверх их проихзвдеения переводит сигнал в домен вероятностей:

$$p(w_O \mid w_I) = \frac{\exp(\mathbf{u}_{w_O}^{\top}\mathbf{v}_{w_I})}{\sum_{w \in V}\exp(\mathbf{u}_{w}^{\top}\mathbf{v}_{w_I})}$$

Здесь, также как в модели Бенжио, мы должны оценивать релевантность пар слов, поэтому возникает та же проблема вычисления знаменателя в софтмаксе. Для быстрого вычисления авторы использовали уже знакомый нам иерархический софтмакс. Единственное отличие - они брали дерево Хаффмана (структуру, учитывающкю частоту слова). А в более поздней версии перешли на негативное сэмплирование. 

Архитектуры CBOW и Skip-gram в сочетании с иерархическом софтмаксом и прореживанием частотных слов позволили обучить качественные векторы на корпусе из 1,6 млрд слов меньше чем за сутки на обычных процессорах. Не менее важным оказалось то, что авторы открыли исходный код и выложили готовые векторы, обученные на корпусе Google News.

### Сэмплирование негативов
Список оптимизаций расчета софтмакса можно дополнить ещё одним важным приёмом, который с лёгкой руки Миколова также стал стандартом. Негативное сэмплирование (Negative Sampling) - это не столько прием оптимизации, сколько режим обучения. Идея в том, чтобы решать задачу не многоклассовой классификации ($\text{"слово"} \rightarrow \text{"контекст"}$), а заменить её задачей бинарной классификации ($\text{"слово + контекст"} \rightarrow \text{"да/нет"}$). 

Иными словами, при вычислении функции потерь мы теперь считаем не полный софтмакс, требующий прохода по всем |V| словам, а оцениваем релевантность всего нескольких пар слов: одной настоящей пар (слово + контекст = да) и k фейковых (слово + контекст = нет). Фейковые слова, они же "шум", выбираются случайно из словаря V. Для оценки точности попадания используется логистическая функция потерь:

$$\mathcal{L}_{\text{NEG}} = \underbrace{
-\log \sigma(\mathbf{u}_{w_O}^\top \mathbf{v}_{w_I})}_{\text{ошибка правильной пары}}
\quad \underbrace{-\sum_{i=1}^{k}
\log \left( 1 - \sigma(\mathbf{u}_{w_i}^\top \mathbf{v}_{w_I}) \right)}_{\text{ошибки рандомных пар}} \rightarrow \min
$$

Релевантные пары должны давать сигнал близкий к единице, а с "шумом" к нулю. Такой переход смещает акцент с вычисления точных вероятностей на правильное ранжирование.

Что касается генерации фейковых пар, Миколов обнаружил, что лучше всего работает сэмплирование не из равномерного распределения, а из униграммного, причем из *сглаженного* униграммного распределния. Униграмное распределение - это гистограмма частот всех слов в корпусе: часто встречающиеся слова имеют большую вероятность быть выбранными, чем редкие слова. Возведение в степень уменьшает самые большие вероятности и увеличивает самые малые, а нормировка нужна, чтобы сделать из гистограммы обратно полноценное распределение. Значение 3/4 было подобрано экспериментально.

$$P_{\text{neg}}(w) = \frac{U(w)^{3/4}}{\sum_{w' \in V} U(w')^{3/4}}$$

### Семантическая арифметика
Метод Word2Vec во многом обязан своей популярностью удивительному свойству распределенных представлений, на которое сначала не обратили внимание даже авторы - это возможность производить осмысленные векторные операции (сложение и вычитание) прямо над представлениями слов. Классическим примером, иллюстрирующим это свойство, является знаменитое соотношение: $$\text{«Король»} − \text{«Мужчина»} + \text{«Женщина»} ≈ \text{«Королева»}$$

Допустим, мы обучили Word2Vec модель и на выходе имеем сгенерированные векторные представления всех слов. Тогда, если мы возьмем вектор слова "король", вычтем из него вектор слова "мужчина" и прибавим вектор слова "женщина", то результирующий вектор окажется ближе всего к вектору слова "Королева". То есть модель самостоятельно, изолирует абстрактный признак «королевская власть» от признака «гендер» и способна переносить этот признак на другие объекты. Более того, такие закономерности не ограничивается гендерными парами, векторная арифметика прекрасно работает с географией (например, «Париж» − «Франция» + «Германия» ≈ «Берлин»), грамматическими формами («шёл» − «идти» + «играть» ≈ «играл») и даже степенями сравнения («быстрее» − «быстрый» + «высокий» ≈ «выше»).

Важно отметить, что модель нигде явно не обучается складывать и вычитать слова, это классический пример *эмерджентного* свойства (то есть возникающего на большом масштабе). Просто при обучении нейросеть вынуждена оптимально структурировать смыслы, в результате чего возникают аналогии с человечксой логикой. У модели как будто появляется "понимание" языка.

Главное ограничение — полисемия (многозначность слов). Поскольку в Word2Vec каждому токену соответствует ровно один вектор, смыслы разных значений слова «схлопываются» в одну точку. Например, вектор слова «замок» будет представлять собой усредненное значение между средневековым строением и дверным механизмом и теряет интерпертируемость. Кроме того, для правильного расположения, нужна большая выборка. Если слово встречается редко, модель не успевает точно определить его координаты, из-за чего аналогичные математические сдвиги на нем «сбоят». Также модель отражает культурные и социальные стереотипы, содержащиеся в текстах (например, предвзятость при переносе профессий), что может искажать логику аналогий.

## Модель GloVe
Итого, к 2014 году в индустрии появилось две группы подходов, использующихся для кодирования текстов. Первая - это методы типа латентно-семантического индексирования (LSI), основанные на декомпозиции глобальной матрицы частот. Вторая - это параметрические модели типа Word2Vec, моделирующие представления слов через их совместное использование в одном локальном контексте. Команда [(Pennington et al., 2014)](https://aclanthology.org/D14-1162/) из Стенфорда попыталась объединить обе идеи в единый гибридный подход, взяв сильные стороны из каждой. Модель назвали **GloVe** (Global Vectors). От глобальных . От локальных

Идея метода в том, что строится матрица совместной встречаемости слов (co-occurence matrix), посчитанная на базе корпуса текстов. В ячейках матрицы $X_{ij}$ хранится, сколько раз слово номер $j$ встречается рядом (в том же контексте) со словом $i$.

Ключевое наблюдение авторов: осмысленную информацию несут не сами по себе совместные встречаемости, а их отношения. Отношение вероятностей $P_{ik}/P_{jk}$, где $P_{ik} = X_{ik}/\sum_l X_{il}$ — вероятность встретить пробное слово $k$ рядом со словом $i$, хорошо разделяет релевантные и нерелевантные ассоциации: оно велико, если $k$ ближе к $i$, чем к $j$, и мало в обратном случае. 

Из этого наблюдения выводится взвешенная задача наименьших квадратов, в которой скалярное произведение векторов приближает логарифм числа совместных встречаемостей:

$$J = \sum_{i,j=1}^{V} f(X_{ij})\,\big(\mathbf{w}_i^{\top}\tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\big)^2.$$

Здесь $\mathbf{w}_i$ и $\tilde{\mathbf{w}}_j$ — векторы слова и контекстного слова, $b_i, \tilde{b}_j$ — смещения, а весовая функция $f$ гасит вклад как слишком редких, так и чересчур частых пар:

$$f(x) = \begin{cases} (x/x_{\max})^{\alpha}, & x < x_{\max} \\ 1, & x \ge x_{\max} \end{cases}$$

с типичными $x_{\max} = 100$ и $\alpha = 3/4$.

GloVe обучается не проходом по тексту скользящим окном, а по уже собранной матрице статистик, что делает его эффективным и хорошо распараллеливаемым. По качеству на задачах аналогий и сходства слов он оказался сопоставим с Word2Vec.

## Модель FastText
В 2016 году Томаш Миколов, автор оригинального Word2Vec, работал в Facebook AI Research и вместе с коллегами они переработали модели Word2Vec и GloVe, у которых они видели два главных ограничения. Во-первых те модели полностью игнорировали морфологию: слова «бежать», «бежит», «убежал» кодируются разными векторами, хотя очевидно родственны. Что особенно критично для морфологически богатых языков, таких как, например, русский. Во-вторых, модели беспомощны перед ранее не встречавшимися словами (out-of-vocabulary, OOV): для этих слов свой эмбединг просто не предусмотрен. Не то, чтобы это было частой проблемой, но тем не менее она иногда встречается, например, в текстах со специальной лексикой.

[(Joulin, 2016)](https://arxiv.org/abs/1607.01759)

Чтобы побороть эти ограничения, авторы перевели модель с уровня слов на уровень отдельных символов и назвали ее **FastText** [(Bojanowski et al., 2016)](https://arxiv.org/abs/1607.04606), Модель обучения оставили ту же, Skip-gram, но теперь каждое слово описывается как набор *символьных* n-грамм - подстрок, сотоящих из трёх–шести символов. 

Каждой такой n-грамме $g$ сопоставляется свой вектор $\mathbf{z}_g$. Оценка совместимости анализируемого слова $w$ (с множеством его n-грамм $\mathcal{G}_w$) и контекстного слова $c$ считается не как одно скалярное произведение, а как сумма по n-граммам:

$$s(w, c) = \sum_{g \in \mathcal{G}_w} \mathbf{z}_g^{\top}\mathbf{v}_c$$

Благодаря такому способу разибения модель стала учитывать морфологию: слова с общими корнями и аффиксами разделяют символьные n-граммы, а значит, автоматически получают близкие векторы. Кроме того, для любого нового слова, не встречавшегося при обучении, вектор можно собрать из его символьных n-грамм и проблема out-of-vocabulary фактически снимается. При этом метод остаётся быстрым и обучается на больших корпусах так же эффективно, как Word2Vec.

FastText во многом стал стандартом *статических* эмбеддингов. Дальнейшее равитие идеи дистрибутивных векторов уперлось в проблему независимости этих представлений от контекста. И такая независимость от контекста - самое большое ограничение, для преодоления которого стали разрабатывать целую линейку более современных подходов, сначала на базе рекурсивных моделей (подробнее в главе 3), а потом Трансформерных (подробнее в главе 4).

Важно отметить, что FastText это не только модель эмбедингов, но и разработанная библиотека с методами классификации. Если в течение нескольких лет после своего выхода FastText был популярен скорее как источник эмбедингов, то в дальнейшем он стал стандартом именно в области классикации и предобатки больших корпусов текстов. Библиотеку до сих пор активно используют, когда нужно отфильтровать текст (например, см. процесс обучения модели DeepSeek-Math) или определить его язык (модель lid.176 отраслевой стандарт).

### О природе эмбедингов
Вскоре после выхода Word2Vec многочисленные команды и исследователи начали активно изучать, почему это работает, искать теоретическое обоснование.

В работе [(Levy et al., 2014)](https://aclanthology.org/W14-1618.pdf) они изучали вопрос, уникальна ли для нейросетевых эмбеддингов способность решать аналогии через арифметику векторов? Оказалось, что не уникальна. Аналогии неплохо решаются и в явном разреженном пространстве, где каждое измерение — это просто PMI с конкретным контекстным словом.

Семантическая арифметика — это не исключительное свойство Word2Vec, а скорее свойство "контекстных" семантических моделей. Старые глобальные методы типа LSI и новые глубокие контекстные модели (LLM) классической векторной арифметикой «из коробки» не обладают.

В дальнейшем мы увидим эволюцию эмбедингов в сторону контекстуализированных представлений - таких, где представление зависит не только от слова, но и от всего его контекста. И хотя геометрия таких моделей все еще сохраняет способность к линейным аналогиям, простые операции сложения и вычитания статичных векторов гораздо меньше применимы на практике.

В работе [(Levy et al, 2015)](https://aclanthology.org/Q15-1016.pdf) решили сравнить, какая модель фундмаентально лучше. Оказалось, что все методы семейства примерно сопоставимы по качеству, а наблюдаемые ранее различия обусловлены просто выбором инженерных настроек, таких как грамотная предобратока, выбор контекстного окна и генерация итоговых векторов.

Каждый гитарный усиилтель обладают своим уникальным характером. Но если хочется нарулить тот самый звук, как на записи любимой группы, почти всегда это можно сделать, просто тщательно покрутив ручки настроек.

### Интерпретация векторной геометрии
В 2014 году [(Levy et al)](https://proceedings.neurips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf) дали формальное обоснование геометричности представлений. Они показали, что процесс обучения Word2Vec эквивалентен факторизацией матрицы взаимной информации слов (Pointwise Mutual Information). 

Точечная взаимная информация (Poinwise Mutual Information) - это классическая мера из теории информации, которая отображает степень ассоциированности двух случайных величин. Она показывает, насколько конкретная пара слов встречается чаще по сравнению со случаем, если бы они были полностью независимы. Метрику не стоит путать просто со взаимной информацией (Mutual Information), которая представляет собой матожидание PMI по всем возможным парам слов.

$$PMI(w, c) = \log \frac{P(w, c)}{P(w) \cdot P(c)} = \log \bigg[ \frac{f(w, c) / N}{f(w) / N \cdot f(c) / N} \bigg]$$

Этот факт проложил мостик между Word2Vec и частотными (глобальными) методами типа LSI - оказалось, что оба метода решают очень похожую задачу.

Кроме того, этот результат помог проинтерпретировать ту самую семантическую геометрию Word2Vec. Рассмотрим какое-нибудь слово, относительно которого сможем оценивать контекстную близость, Пусть это будет слово $c = \text{«корона»}$. Делаем предположение, что в реальных текстах $P(\text{король}, c) / P(\text{мужчина}, c) = P(\text{королева}, c) / P(\text{женщина}, c)$. То есть короли носят короны во много раз чаще обычных мужчин, а королевы — во столько же раз чаще обычных женщин.

Возьмем логарифм от обеих дробей и разложим их по свойству логарифма в сумму $\log P(\text{король}, c) - \log P(\text{мужчина}, c) = \log P(\text{королева}, c) - \log P(\text{женщина}, c)$. Эти логарфимы - почти и есть та самая взаимная информация, только с точностью но нормировки, которая сокращается.

Выше мы выяснили, что алгоритм Word2Vec своим скалярным произведением аппроксимирует взаимную информацию ($\vec{w} \cdot \vec{c} \approx PMI(w,c)$), поэтому приближенно заменяем логарифмы скалярным произведением. И сократив с обоизх сторон общий контекст $\vec{c}$, мы получаем строго параллельные семантические сдвиги (так просто сокращать вектор $\vec{c}$ не совсем корректно, но учитывая, что обучающая выборка большая, приближенно можем так сделать).
$$\vec{\text{Король}} - \vec{\text{Мужчина}} = \vec{\text{Королева}} - \vec{\text{Женщина}}$$

## Другие модели эмбедингов
Слово «эмбеддинг» плотно вошло в повседневный словарь исследователей и инженеров, а предобученные векторы начали подставлять на вход практически любой модели, работающей с текстом, и это давало заметный прирост качества. Например, [(Kim, 2014)](https://arxiv.org/pdf/1408.5882) в своей работе тестироавл использование свёрточной нейронной сети для классификации предложений. Добавление предобученных word2vec эмбедингов вместо случайной инициализации повышала точность на некоторых бенчмарках до 5 пп.

Кроме того, возникла целая волна моделей с суффиксом «vec»: Doc2Vec (Paragraph Vector) для кодирования документов, sent2vec для кодирования предложений, DeepWalk и node2vec для вершин графов, graph2vec для графов целиком, item2vec и prod2vec для товаров в рекомендательных системах. Оказалось, что идею «объект определяется своим контекстом» можно перенести на любые последовательности и отношения.

__Def2Vec__. 
Напомним, что в классическом латентно-семантическом индексировании (LSI), мы берем большой корпус текстов, например, статей Википедии, строим по нему матрицу частот и раскладываем её в произведение скрытых представлений, обычно методом SVD. [(Morazzoni et al., 2023)](https://aclanthology.org/2023.icnlsp-1.21.pdf) решили вместо большого корпуса текстов брать небольшие выжимки из толкового словаря с определением анализируемого слова и строить Term-Document матрицу только на базе этих определений.  Идея в том, что словарное определение — это сжатая, целенаправленная формулировка смысла слова, содержащая именно те слова-маркеры, которые максимально точно характеризуют его значение. 

Модель назвали __Def2Vec__ (Definition To Vector). На тестах она показала результаты, сопоставимые по качеству с Word2Vec и GloVe, при этом основная выгода здесь в более дешевом препроцессинге - нет необходимсоти собирать и очищать большие коллекции документов, их можно загрузить сразу из уже готового словаря. Кроме того, мы не зависим от втсречаемости слова в обучающем корпусе - если словарь достаточно полный, слово в нём скорее всего есть. Таким образом частично решается проблема OOV (out-of-vocabulary), работа с не встречавшимися ранее словами. И даже если слово появляется позже, его эмбединг можно сгенерировать "находу" без необхоимости переобучать всю модель (в машинном обучении динамическое обновление SVD разложения называют folding-in).

__LexVec__. [(Salle et al, 2016)](https://arxiv.org/abs/1606.00819) в своей модели __LexVec__ также взяли за основу идею латентно-семантического индексирование (LSI), но предложили два существенных изменения. Во-первых, они заменили матрицу, которую декомпозируют - вместо сырой матрицы частот (Term-Document) стали использовать матрицу взаимной информации между словами (Positive Pointwise Mutual Information, PPMI), которая нам уже встречалась в модели Word2Vec. Напомним, как она считается через вероятность $P(w,c)$ совместного сочетания пары слов $w$ и $c$ (или соотвественно частоту $f(w,c)$):

$$PMI(w, c) = \log \frac{P(w, c)}{P(w) \cdot P(c)} = \log \frac{f(w, c) / N}{\frac{f(w)}{N} \cdot \frac{f(c)}{N}}$$

Во-вторых, они заменили ресурсозатратное матричное разложение SVD на более легковесную итеративную оптимизацию методом градиентного спуска. Оптимизруется тот же функционал, что в Word2Vec, позитивные примеры - это пары слов из одного конектста, а негативные сэмплируются случайно. Если два слова взамиоисключающие, и соотвественно встречаются реже случайного отобра, то их взаимная информация будет отрицательной. В реальных матрицах таких пар много и они не особо полезны, поэтому авторы решили орицательные значение выкидывать.

__BEAST__.
[(Palomo-Alonso et al., 2026)](https://www.techrxiv.org/doi/pdf/10.36227/techrxiv.177220385.59586322/v1?redirectToLatest=false) Если FastText работает на уровне символьных н-граммов, то модель __BEAST__ (Byte-wise Embedding Architecture for Semantic Transfer) идет дальше и работает на уровне отдельных байтов. Модель обучается через дистилляцию из преобученной большой языковой модели, например, GPT-3. Она учится генерировать эмбединги байтов так, чтобы их комбинация приблизительно совпадала с эмбедингом всего токена. Комбинирование обычно делается через усреднение.

__Эмбединги Пуанкаре.__
[(Nickel et al., 2017)](https://arxiv.org/abs/1705.08039) предложили вместо встраивания эмбедингов в Евклидово пространство $\mathbb{R}^N$, как это обычно делается, генерировать их в гиперболическом пространстве. Пример такого пространства - это *шар Пуакнкаре*, который отображает Евклидлово пространство в обычный шар единичного радиуса. Идея в том, что гиперболическое пространство лучше подходит для моделирования множеств, имеющих иерархическую природу, таких как, например, таксономии, графы синонимов WordNet и прочее. Корень такого дерева может располагаться в центре пространства, а ноды распределяются вокруг него. Всё оставльное, включая постановку задачи, заимствуется от Word2Vec.

__StarSpace__.
[(Wu et al., 2017)](https://arxiv.org/abs/1709.03856) сделали попытку расширить идею Word2Vec до более универсальной модели. В рамках StarSpace представления всё так же моделируют контекстную близость, но в роли объекта близости могут быть любые сущности - не только слова, но и документы, пользователи, товары. А в роли контекста - любые признаки, метка класса, купленный товар и т.д.

__SenseGram__. В числе ограничений модели Word2Vec мы отмечали, что такие эмбединги конектно-независимые, то есть задаются для отдельного слова $\big( \, \text{"ключ"} \rightarrow (x_1 ... x_n) \, \big)$, но не для набора слов $\big( \, \text{"(горный) ключ"} \rightarrow (x_1 ... x_n) \, \big)$. Это значит, что если у слова несколько значений, то нельзя выделить наиболее подходящее. Это может быть большим ограчниением для слов допускающих полисемию или омонимию. 

[(Pelevina et al., 2016)](https://arxiv.org/abs/1708.03390) свою модель SenseGram, в рамках которой эмбединг слова предсмтавляется в виде комбинации эмбедингов значений. В этом методе нет классического обучения, мы берем предобученный эмбединг слова и смотрим на его соседей в признаковом пространстве. Предполагается, что среди этих соседей есть примеры всех семантик, которые раздляет слово. Например, для слова ключ это могут быть ("родник, горный"), ("музыка, басовый") и ("дверь, открыть"). Затем эти соседи кластеризуются в K однородных групп (k определяется динамически) и в качестве представления слова используется набор из K векторов, представляющих центры этих групп. Таким образом моделируется мультисемантичность каждого слова.

Для кластеризации используется алгоритм *Chinese Whispers*, который предназначен для кластеризации графов. Его суть в том, что строится граф ближайших соседей для множества объектов. Затем каждое слово размечается в ту метку, которая проставлена у большинства соседей.

## Эмбеддинги для русского языка

[опциональный раздел]

Для русского языка создан ряд проектов, предоставляющих предобученные статические векторные представления слов, основанные на классических алгоритмах Word2Vec, GloVe и FastText. Ниже приведены наиболее значимые из них с указанием хронологии развития.

Проект __RusVectōrēs__ был основан в 2015 году и задумывался как стандарт векторно-семантических моделей для русского языка. Там появились Word2Vec модели. Спустя 4 года расширили  WordwVec и FastText моделями, обученными на русской Википедии, новостных корпусах Lenta.ru и доугих текстах. В 2021 годау модели обучили на Национальном корпусе русского языка (НКРЯ) и дампе русской Википедии. Модели доступны на официальном сайте проекта: rusvectores.org.

Библиотека __DeepPavlov__ начала публиковать предобученные векторы для русского языка в 2017 году. Модели на основе FastText (skip-gram) и GloVe были обучены на совместном корпусе русской Википедии и Lenta.ru, все векторы имеют размерность 300. Помимо этого, были опубликованы модели, обученные на корпусе русского Твиттера, что делает их полезными для задач, связанных с разговорной речью.

__Navec__ — это набор компактных GloVe-эмбеддингов для русского языка, созданный в рамках проекта Natasha. Первый релиз библиотеки Navec состоялся 29 октября 2020 года. Модель содержит около 500 000 слов и 300 измерений, а её размер составляет примерно 50 МБ благодаря применению квантизации — замене 32-битных чисел на 8-битные коды. Исходные тексты для обучения были взяты из проекта RUSSE, объём корпуса составил 145 ГБ художественной литературы. Проект доступен на GitHub: natasha/navec.

__RDT__ (Русский дистрибутивный тезаурус) представляет собой первый свободно доступный дистрибутивный тезаурус русского языка. Модель эмбеддингов была опубликована 18 марта 2017 года. Обучение проводилось с использованием алгоритма SGNS (Word2Vec) на корпусе книг объёмом 12,9 млрд слов, собранном из библиотеки lib.rus.ec. Векторы имеют размерность 500, размер контекстного окна — 10 слов.

Репозиторий __NLPL__ (Nordic Language Processing Laboratory) предоставляет доступ к моделям эмбеддингов для множества языков, включая русский. Модель word2vec-ruscorpora-300 была обучена на полном Национальном корпусе русского языка объёмом около 270 млн токенов. Векторы имеют размерность 300, размер окна — 2, применялись лемматизация и POS-теггинг. Версия репозитория 2.0, в которую вошла эта модель, была опубликована 27 декабря 2019 года. Доступ к моделям осуществляется через сайт vectors.nlpl.eu.

Библиотека __Gensim__ предоставляет простой API для загрузки предобученных моделей, включая word2vec-ruscorpora-300 — модель Continuous Skip-gram, обученную на полном Национальном корпусе русского языка объёмом около 250 млн слов. Модель содержит 184 973 вектора размерностью 300, размер окна — 10, применялась лемматизация и разметка Universal PoS. Загрузка выполняется одной командой: `api.load("word2vec-ruscorpora-300")`.

__OpenSubtitles__. Коллекция subs2vec включает эмбеддинги для русского языка, обученные на корпусе субтитров OpenSubtitles 2018 с использованием алгоритма FastText (skip-gram). Публикация датасета состоялась в 2020 году. Векторы имеют размерность 300 и обучены на естественном языке фильмов и телепередач, что делает их полезными для задач, связанных с разговорной речью. Доступны различные конфигурации с разными размерами окна (от 2 до 10) и размерностями (от 100 до 500).